## Libraries import

In [12]:
import os
from PIL import Image
from torch.utils.data import Dataset, Subset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import pandas as pd


## Dataset Class

In [13]:
class ResilienceDataset(Dataset):
    def __init__(self, identity_file_path, attack_status_file_path, attack_info_file_path, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform

        self.identity_df = pd.read_csv(identity_file_path, sep=' ', header=None,
                                     names=['image_name', 'identity'])
        self.unique_identities = self.identity_df['identity'].unique()
        self.identity_to_label = {identity: idx for idx, identity in enumerate(self.unique_identities)}


        self.attack_status_df = pd.read_csv(attack_status_file_path, sep=' ', header=None,
                                          names=['image_name', 'is_attacked'])

        self.attack_df = pd.read_csv(attack_info_file_path, sep=' ', header=None,
                                        names=['attacked_image', 'original_identity', 
                                               'attack_type', 'target_image', 'attack_id'])
        self.attack_info_map = {}
        for _, row in self.attack_df.iterrows():
                self.attack_info_map[row['attacked_image']] = {
                    'attack_type': row['attack_type'],
                    'target_image': row['target_image'],
                    'attack_id': row['attack_id']
                }

        merged_df = pd.merge(self.identity_df, self.attack_status_df, on='image_name', how='inner')
        self.combined_data = []

        for _, row in merged_df.iterrows():
            sample = {
                'image_name': row['image_name'],
                'identity': row['identity'],
                'is_attacked': row['is_attacked'] == 1
            }
        
            if sample['is_attacked'] and row['image_name'] in self.attack_info_map:
                attack_details = self.attack_info_map[row['image_name']]
                sample['attack_type'] = attack_details['attack_type']
                sample['target_image'] = attack_details['target_image']
                sample['attack_id'] = attack_details['attack_id']
                if attack_details['attack_type'] == 'impersonation':
                    target_img = attack_details['target_image']
                    target_matches = self.identity_df[self.identity_df['image_name'] == target_img]
                    if not target_matches.empty:
                        sample['target_identity'] = target_matches.iloc[0]['identity']
                    else:
                        sample['target_identity'] = -1
            else:
                sample['attack_type'] = "none"
                sample['target_image'] = ""
                sample['target_identity'] = -1
                sample['attack_id'] = -1
            
            self.combined_data.append(sample)
        
        self.num_identities = len(self.unique_identities)


    def __len__(self):
        return len(self.combined_data)

    def __getitem__(self, idx):
        sample = self.combined_data[idx]
        img_path = os.path.join(self.image_dir, sample['image_name'])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        identity_label = self.identity_to_label[sample['identity']]
        target_label = -1
        if sample['is_attacked'] and sample['attack_type'] == 'impersonation' and sample['target_identity'] is not None:
            target_label = self.identity_to_label[sample['target_identity']]
        
        result = {
            'image': image,
            'identity': identity_label,
            'is_attacked': 1 if sample['is_attacked'] else 0,
            'attack_type': sample['attack_type'],
            'target_identity': target_label,
            'attack_id': sample['attack_id'],
            'image_name': sample['image_name']
        }
        
        return result


## Data Loading & Transform

In [14]:
def create_dataloaders_with_partition(identity_file, partition_file, attack_status_file,
                                     attack_info_file=None, image_dir='images',
                                     batch_size=32):
    transform = transforms.ToTensor()

    train_dataset = ResilienceDataset(
        identity_file_path=identity_file,
        attack_status_file_path=attack_status_file,
        attack_info_file_path=attack_info_file,
        image_dir=image_dir,
        transform=transform
    )

    partition_df = pd.read_csv(partition_file, sep=' ', header=None,
                              names=['image_name', 'partition'])
    
    train_indices = []
    val_indices = []
    test_indices = []

    image_to_idx = {}
    for idx, item in enumerate(train_dataset.combined_data):
        image_to_idx[item['image_name']] = idx
    
    for _, row in partition_df.iterrows():
        img_name = row['image_name']
        partition = row['partition']
        
        if img_name in image_to_idx:
            idx = image_to_idx[img_name]
            if partition == 0:
                train_indices.append(idx)
            elif partition == 1:
                val_indices.append(idx)
            elif partition == 2:
                test_indices.append(idx)

    train_subset = Subset(train_dataset, train_indices)
    val_subset = Subset(train_dataset, val_indices)
    test_subset = Subset(train_dataset, test_indices)

    train_loader = DataLoader(
        train_subset, batch_size=batch_size, shuffle=True
    )
    
    val_loader = DataLoader(
        val_subset, batch_size=batch_size, shuffle=False
    )
    
    test_loader = DataLoader(
        test_subset, batch_size=batch_size, shuffle=False
    )

    return train_loader, val_loader, test_loader, train_dataset

In [15]:
base_path = '../AdvCelebA'
identity_file = os.path.join(base_path, 'identity_CelebA.txt')
partition_file = os.path.join(base_path, 'list_eval_partition_no_overlap.txt')
attack_status_file = os.path.join(base_path, 'attack_CelebA.txt')
attack_info_file = os.path.join(base_path, 'final_attack_attackid_cw.txt')
image_dir = os.path.join(base_path, 'images')

train_loader, val_loader, test_loader, dataset = create_dataloaders_with_partition(
        identity_file=identity_file,
        partition_file=partition_file,
        attack_status_file=attack_status_file,
        attack_info_file=attack_info_file,
        image_dir=image_dir,
        batch_size=32
)

print(f"Number of identities: {dataset.num_identities}")
print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Testing samples: {len(test_loader.dataset)}")


Number of identities: 10177
Training samples: 81271
Validation samples: 10132
Testing samples: 10005


In [16]:
for batch in train_loader:
    print(f"Batch size: {batch['image'].shape}")
    print(f"Identity labels: {batch['identity']}")
    print(f"Is attacked: {batch['is_attacked']}")
    print(f"Attack types: {batch['attack_type']}")
    break

Batch size: torch.Size([32, 3, 112, 112])
Identity labels: tensor([2991, 1764, 5110, 1341, 2763, 4213, 5207, 5826, 1903,  977, 6097, 7993,
        2095, 3998, 4259, 2840, 4260, 1714, 1462, 3924, 1940, 6355, 5458, 2522,
        4968, 1217, 4698, 4707, 2668, 4047, 2017, 3123])
Is attacked: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])
Attack types: ['none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none', 'none']


## Model

In [17]:
class FaceRecognitionModel(nn.Module):
    def __init__(self, num_classes):
        super(FaceRecognitionModel, self).__init__()
        weights = MobileNet_V2_Weights.DEFAULT
        base_model = mobilenet_v2(weights=weights)
        self.features = base_model.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.embedding = nn.Linear(base_model.last_channel, 256)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).view(x.size(0), -1)
        embedding = self.embedding(x)
        logits = self.classifier(embedding)
        return logits, embedding

## Loss Function

In [18]:
criterion = nn.CrossEntropyLoss()

## Accuracy Function

In [19]:
def accuracy(preds, labels):
    _, predicted = torch.max(preds, 1)
    correct = (predicted == labels).sum().item()
    return correct / labels.size(0)

## Train Loop

In [20]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0

    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(device)
            labels = batch['identity'].to(device)

            logits, _ = model(images)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            total_acc += accuracy(logits, labels)

    return total_loss / len(dataloader), total_acc / len(dataloader)

In [21]:
def train_and_validate(model, train_loader, val_loader, optimizer, criterion, device, epochs=10):
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_acc = 0.0

        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]", leave=False)

        for batch in train_bar:
            images = batch['image'].to(device)
            labels = batch['identity'].to(device)

            optimizer.zero_grad()
            logits, _ = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_acc += accuracy(logits, labels)

        avg_train_loss = total_loss / len(train_loader)
        avg_train_acc = total_acc / len(train_loader)

        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f}, Acc: {avg_train_acc:.4f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')
            print("✅ Saved new best model")

    print(f"📈 Best validation accuracy: {best_val_acc:.4f}")

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FaceRecognitionModel(num_classes=dataset.num_identities).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

train_and_validate(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=10
)

Epoch 1/10 | Train Loss: 8.6056, Acc: 0.0023 | Val Loss: 12.9125, Acc: 0.0000


Epoch 2/10 | Train Loss: 6.9971, Acc: 0.0226 | Val Loss: 15.2752, Acc: 0.0000


KeyboardInterrupt: 